In [1]:
import re
import pandas as pd
from random import choice

from ruwordnet import RuWordNet

from pymystem3 import Mystem

from pymorphy3 import MorphAnalyzer

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('ru_cefr_short.csv')
df

,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [3]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

In [4]:
texts_to_augment = []

for i in range(len(train_labels)):
  if train_labels[i] == 6:
    texts_to_augment.append(train_texts[i])


len(texts_to_augment)

120

In [5]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

def detokenize(tokens):
    text = ""
    for i, tok in enumerate(tokens):
        if i == 0:
            text += tok
            continue

        if tok in ",.!?:;)]}»":
            text += tok
        elif tokens[i - 1] in "([«":
            text += tok
        else:
            text += " " + tok
    return text

In [6]:
m = Mystem()

def lemmatize(text):
  lemmas = m.lemmatize(text)
    
  lemmas = [i for i in lemmas if i != ' ' and i != '\n']

  return lemmas

In [7]:
morph = MorphAnalyzer()

GOOD_POS = {
    "NOUN",   
    "ADJF", 
    "ADJS",
    "VERB",   
    "INFN"   
}

def is_replaceable_word(word):
    parses = morph.parse(word)
    if not parses:
        return False

    p = parses[0]
    pos = p.tag.POS

    if pos not in GOOD_POS:
        return False

    # исключаем местоимения и местоименные слова
    if pos == "NPRO":
        return False

    # исключаем местоименное прилагательное: этот, мой, какой, всякий
    if "Apro" in p.tag:
        return False

    return True

In [8]:
morph = MorphAnalyzer()

wn = RuWordNet("/home/tyumen/baseline_libs/myenv/lib/python3.12/site-packages/ruwordnet/static/ruwordnet-2021.db")

clean_title = lambda title: re.sub(r'\s*\([^)]*\)', '', title).strip()

def get_synonyms(word):
    parse = morph.parse(word)[0]
    lemma = parse.normal_form
    
    synonyms = []
    
    for sense in wn.get_senses(lemma):
        for synonym_name in sense.synset.senses:
            synonym = synonym_name.name
            synonym = synonym.lower()
            if " " not in synonym and synonym != lemma:
                synonyms.append(clean_title(synonym))
    return synonyms

In [9]:
with open('new_vocab_a1.txt', 'r'):
    file = open('new_vocab_a1.txt')
    a1 = file.readlines()

with open('new_vocab_a2.txt', 'r'):
    file = open('new_vocab_a2.txt')
    a2 = file.readlines()

with open('new_vocab_b1.txt', 'r'):
    file = open('new_vocab_b1.txt')
    b1 = file.readlines()

with open('new_vocab_b2.txt', 'r'):
    file = open('new_vocab_b2.txt')
    b2 = file.readlines()

with open('new_vocab_c1.txt', 'r'):
    file = open('new_vocab_c1.txt')
    c1 = file.readlines()


a1 = set([w.replace('\ufeff', '').strip() for w in a1 if w.strip()])
a2 = set([w.replace('\ufeff', '').strip() for w in a2 if w.strip()])
b1 = set([w.replace('\ufeff', '').strip() for w in b1 if w.strip()])
b2 = set([w.replace('\ufeff', '').strip() for w in b2 if w.strip()])
c1 = set([w.replace('\ufeff', '').strip() for w in c1 if w.strip()])

print(len(a1), len(a2),len(b1), len(b2), len(c1))


only_a1 = a1
only_a2 = a2 - a1
only_b1 = b1 - a2 - a1
only_b2 = b2 - b1 - a2 - a1
only_c1 = c1 - b2 - b1 - a2 - a1

print(len(only_a1), len(only_a2), len(only_b1), len(only_b2), len(only_c1))

923 1493 2821 5867 11961
923 580 1334 3142 6441


In [10]:
def rank_synonyms(synonyms):
    c1 = []
    b2 = []
    b1 = []
    a2 = []
    a1 = []
    
    for synonym in synonyms:
        if synonym in only_c1:
            c1.append(synonym)
        elif synonym in only_b2:
            b2.append(synonym)
        elif synonym in only_b1:
            b1.append(synonym)
        elif synonym in only_a2:
            a2.append(synonym)
        elif synonym in only_a1:
            a1.append(synonym)

    ranked_synonyms = c1 + b2 + b1+ a2+ a1
    return ranked_synonyms

In [11]:
INFLECTABLE_GRAMMEMES = {
    # число
    "sing", "plur",

    # род
    "masc", "femn", "neut",

    # падеж
    "nomn", "gent", "datv", "accs", "ablt", "loct",
    "gen1", "gen2", "acc2", "loc1", "loc2", "voct",

    # одуш/неодуш
    "anim", "inan",

    # глагольные категории
    "past", "pres", "futr",
    "1per", "2per", "3per",
    "indc", "impr",

    # вид
    "perf", "impf",

    # для кратких/полных прилагательных и т.п.
    "Supr", "COMP"
}

def preserve_case(source, target):
    if source.isupper():
        return target.upper()
    if source[0].isupper():
        return target.capitalize()
    return target

def inflect_synonym_like(source_word, ranked_synonyms):
    source_parses = morph.parse(source_word)
    if not source_parses:
        return None
    source_parse = source_parses[0]

    for synonym_lemma in ranked_synonyms:
        synonym_parses = morph.parse(synonym_lemma)
        if  synonym_parses:
            source_pos = source_parse.tag.POS
        
            synonym_parse = None
            for p in synonym_parses:
                if p.tag.POS == source_pos:
                    synonym_parse = p
                    break
        
            if synonym_parse is None:
                synonym_parse = synonym_parses[0]
        
            source_grammemes = set(source_parse.tag.grammemes)
            needed_grammemes = source_grammemes & INFLECTABLE_GRAMMEMES
        
            inflected = synonym_parse.inflect(needed_grammemes)

            if inflected:
                return preserve_case(source_word, inflected.word)
        

    return None

In [15]:
augmented_texts = []
texts = []

LIST_WORDS_TO_REPLACE = []
LEN_WORDS_TO_REPLACE = []
SYNONYMS_COUNT = []
FOUNDED_SYNONYMS = []
RANKED_SYNONYMS = []
CHANGED_SYNONYM = []

for text in texts_to_augment:
    tokens = tokenize(text)
    words_to_replace = {}

    for idx, word in enumerate(tokens):
        if is_replaceable_word(tokens[idx]):
            words_to_replace[idx] = word

    LEN_WORDS_TO_REPLACE.append(len(words_to_replace))
    LIST_WORDS_TO_REPLACE.append(words_to_replace.values())

    words_to_synonyms = {}

    to_print = {}

    syn_count =  []
    founded_syn = []
    ranked_syn = []
    changed_syn = []
    
    for key in words_to_replace.keys():
        synonyms = get_synonyms(tokens[key])
        ranked_synonyms = rank_synonyms(synonyms)

        syn_count.append(len(synonyms))
        founded_syn.append(synonyms)
        ranked_syn.append(ranked_synonyms)

        if len(ranked_synonyms) > 0:
            new_form = inflect_synonym_like(words_to_replace[key], ranked_synonyms)
            if new_form:
                words_to_synonyms[key] = new_form
                changed_syn.append(new_form)
            else:
                changed_syn.append('')

    SYNONYMS_COUNT.append(syn_count)
    FOUNDED_SYNONYMS.append(founded_syn)
    RANKED_SYNONYMS.append(ranked_syn)
    CHANGED_SYNONYM.append(changed_syn)
            
    new_text = []

    for i in range(len(tokens)):
        if i in words_to_synonyms.keys():
            new_text.append(words_to_synonyms[i])
        else:
            new_text.append(tokens[i])

    new_text = detokenize(new_text)

    if new_text != text:
        print(text)
        print(new_text)
        print('\n')

        augmented_texts.append(new_text)
        texts.append(text)

3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
3: 38 – 3: 54 Собственно, сама по себе радиация незаразна. Например, могучими дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат индивидов.


Недавно компания Uber объявила об инвестиции одного миллиарда долларов в собственную систему автоматизированного вождения, а компания Waymo даже запустила приложение на Android для владельцев их самоуправляемых машин.
Недавно фирма Uber объявила об инвестиции одного миллиарда долларов в собственную привычку автоматизированного вождения, а фирма Waymo даже запустила приложение на Android для собственников их самоуправляемых техник.


Множество повестей: «Двойник», «Дядюшкин сон», «Записки из подполья» и другие, рассказы, публицистика, дневники, даже стихи были в творчестве великого автора.
Множество повестей: «Двойник», «Дядюшкин сон», «Запис

In [16]:
stats_df = pd.DataFrame()

stats_df['Список слов для замены'] = LIST_WORDS_TO_REPLACE
stats_df['Количество слов для замены'] = LEN_WORDS_TO_REPLACE
stats_df['Найденные синонимы для каждого слова'] = FOUNDED_SYNONYMS
stats_df['Количество найденных синонимов для каждого слова'] = SYNONYMS_COUNT
stats_df['Отсортированные по уменьшению уровня сложности синонимы для каждого слова'] = RANKED_SYNONYMS
stats_df['Выбранный синоним для каждого слова'] = CHANGED_SYNONYM

stats_df

,Список слов для замены,Количество слов для замены,Найденные синонимы для каждого слова,Количество найденных синонимов для каждого слова,Отсортированные по уменьшению уровня сложности синонимы для каждого слова,Выбранный синоним для каждого слова
0,"(радиация, незаразна, мощными, дозами, излучен...",11,"[[радиоизлучение], [], [могучий, мощнейший, мо...","[1, 0, 4, 0, 1, 0, 1, 0, 1, 12, 16]","[[], [], [могучий], [], [], [], [], [], [], []...","[могучими, индивидов]"
1,"(компания, объявила, инвестиции, миллиарда, до...",15,"[[компашка, фирма], [объявлять, объявлять, огл...","[2, 4, 3, 0, 4, 0, 11, 1, 1, 2, 22, 3, 5, 0, 14]","[[фирма], [объявлять, объявлять], [], [], [], ...","[фирма, , привычку, фирма, , собственников, те..."
2,"(Множество, повестей, Двойник, Дядюшкин, сон, ...",15,"[[куча, масса, тьма, уйма], [], [], [], [греза...","[4, 0, 0, 0, 3, 1, 3, 4, 0, 0, 3, 36, 1, 2, 1]","[[куча, масса, тьма], [], [], [], [], [], [под...","[, , , , пребывали, выдающегося]"
3,"(Встречи, одноклассников, одногруппников, прев...",15,"[[турнир, соревнование, первенство, баталия], ...","[4, 1, 0, 10, 1, 0, 19, 10, 4, 17, 0, 4, 15, 7...","[[турнир, первенство, соревнование], [], [], [...","[, преображаются, пирушки, разрываются, доноси..."
4,"(Самопрезентация, главная, черта, времени, неи...",19,"[[], [основной, наиглавнейший, главнейший], [г...","[0, 3, 13, 6, 3, 9, 16, 10, 19, 1, 2, 8, 8, 7,...","[[], [основной], [рубеж, грань, предел, линия,...","[основная, грань, , неисправимая, патология, И..."
...,...,...,...,...,...,...
115,"(испытания, участников, Жюри, определит, автом...",14,"[[крест, экзамен, проверка], [участница], [], ...","[3, 1, 0, 32, 6, 4, 0, 22, 16, 10, 0, 2, 10, 5]","[[крест, экзамен], [участница], [], [помещать,...","[, , задаст, , видится, индивидов, отвечает, д..."
116,"(Прома, способна, предложить, дилерам, божески...",16,"[[], [], [предлагать, предлагать, внести, внос...","[0, 0, 13, 0, 1, 3, 0, 5, 2, 1, 11, 4, 0, 7, 1...","[[], [], [выдвигать, вносить, предлагать, пред...","[, подходящие, платы, продолжительном, привычк..."
117,"(единственный, игрок, рынке, большинство, круп...",12,"[[единый], [], [базарчик, базар, рыночек, рыно...","[1, 0, 4, 0, 9, 3, 13, 17, 0, 6, 13, 2]","[[единый], [], [базар], [], [внушительный, пор...","[единый, базаре, внушительных, намереваются, ,..."
118,"(крупнейших, стран, поставщиков, высококлассно...",15,"[[порядочный, внушительный, изрядный, широкий,...","[9, 3, 0, 2, 0, 1, 6, 4, 1, 1, 0, 1, 0, 0, 5]","[[внушительный, порядочный, широкий], [держава...","[порядочнейших, держав, первоклассной, межгосу..."


In [17]:
stats_df.to_csv('synonyms_stats.csv')

In [18]:
len(augmented_texts)

119

In [14]:
aug_df = pd.DataFrame()
aug_df['text'] = texts
aug_df['augmented-text'] = augmented_texts
aug_df

,text,augmented-text
0,"3:38–3:54 Собственно, сама по себе радиация не...","3: 38 – 3: 54 Собственно, сама по себе радиаци..."
1,Недавно компания Uber объявила об инвестиции о...,Недавно фирма Uber объявила об инвестиции одно...
2,"Множество повестей: «Двойник», «Дядюшкин сон»,...","Множество повестей: «Двойник», «Дядюшкин сон»,..."
3,Встречи одноклассников и одногруппников превра...,Встречи одноклассников и одногруппников преобр...
4,Самопрезентация — как главная черта времени и ...,Самопрезентация — как основная грань времени и...
...,...,...
114,00:00:24 А сегодня испытания. 13 участников. Ж...,00: 00: 24 А сегодня испытания. 13 участников....
115,«Прома» способна предложить дилерам вполне бож...,«Прома» способна предложить дилерам вполне под...
116,Tesla — не единственный игрок на рынке: больши...,Tesla — не единый игрок на базаре: большинство...
117,"Понятно, что среди крупнейших стран – поставщи...","Понятно, что среди порядочнейших держав – пост..."


In [15]:
aug_df.to_csv('c2_augmented_synonyms.csv')